# Mean Reversion Strategy

A mean-reversion strategy is the **opposite bet** to momentum: it assumes price
moves **overshoot and snap back**. If the market went up last period, we bet it
will come *down* next, and vice-versa. We fade the move instead of following it.

No machine learning and no advanced math — just basic econometrics on the price
series. Financial data is extremely noisy, yet stable statistical patterns hide
inside it, and a simple rule can exploit them. The plan:

1. Turn prices into **log returns**.
2. Use the **previous** period's return as the only signal.
3. Confirm the pattern holds in-sample **and** out-of-sample.
4. Backtest, measure, and subtract realistic **fees**.

> This is the mirror image of the [momentum](01-momentum.ipynb) notebook. The
> single difference is one **minus sign** on the signal (Step 4).

## The data

Daily (`1d`) OHLC bars for **Bitcoin Cash (BCH)**. We know nothing about the
project — only its price history matters. Columns are the usual OHLCV; we build
everything from the **close** (`c`).

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd

# The sample OHLC data ships with the repo under data/samples/. Resolve it
# whether the notebook is launched from the repo root (VS Code default) or from
# its own folder, so "Run All" works either way.
csv_name = 'mean_reversion_ohlc.csv'
csv_path = next(p for p in (Path('data/samples') / csv_name,
                            Path('../../data/samples') / csv_name) if p.exists())

df = pd.read_csv(csv_path)
# The CSV was saved with its row index as an unnamed column — drop it.
del df['Unnamed: 0']
df

## Step 1 — Log returns and the lagged signal

The **log return** `ln(price_t / price_{t-1})` is our building block (logs add up
over time, which makes the equity curve a simple cumulative sum). The **signal**
is the *previous* bar's return — the strategy never looks at the current bar when
deciding a trade.

In [ ]:
df['close_log_return'] = np.log(df['c'] / df['c'].shift(1))
df

In [ ]:
# Yesterday's return, lined up next to today's. Drop the first rows that are NaN.
df['close_log_return_lag_1'] = df['close_log_return'].shift(1)
df = df.dropna()
df

## Encode direction

We reduce each return to its **direction** — `+1` (up / long) or `-1`
(down / short) — for both today's return and yesterday's:

In [ ]:
# +1 if the return is positive, -1 otherwise.
for col in ['close_log_return', 'close_log_return_lag_1']:
    df[f'{col}_dir'] = df[col].map(lambda x: 1 if x > 0 else -1)
df

## Step 2 — Is there a tradable pattern?

Group today's return by **yesterday's direction**. For *mean reversion* we expect
the opposite of momentum: after an up bar the average next return is **negative**,
and after a down bar it is **positive** — the move reverses.

In [ ]:
df.groupby('close_log_return_lag_1_dir').aggregate({'close_log_return': ['mean', 'count', 'sum']})

**Reading it:** the sign of the average return *flips* relative to the prior
direction. Concretely:

1. After the price went **down**, the next bar tends to go **up**.
2. After the price went **up**, the next bar tends to go **down**.

That is mean-reversion behaviour.

## Step 3 — Does it survive out-of-sample?

Split chronologically — first **75%** in-sample, last **25%** out-of-sample — and
require the reversal to appear in both. The holdout stands in for the future: if
the effect is only in-sample, it is noise, not an edge.

In [ ]:
i = int(len(df) * 0.75)
in_sample, out_sample = df.iloc[:i], df.iloc[i:]

in_sample.groupby('close_log_return_lag_1_dir').aggregate({'close_log_return': ['mean', 'count', 'sum']})

In [ ]:
out_sample.groupby('close_log_return_lag_1_dir').aggregate({'close_log_return': ['mean', 'count', 'sum']})

If the reversal persists in the out-of-sample block, the behaviour is stable
over time and worth trading.

## Step 4 — Backtest the rule

Here is the one line that separates mean reversion from momentum: we bet the
**opposite** of the previous bar's direction, i.e. multiply the signal by `-1`.

- `signal = +1` → go long; `signal = -1` → go short.
- Per-bar return earned = `signal * close_log_return`.
- Cumulative sum of those returns = the **equity curve**.

In [ ]:
# Mean reversion: bet AGAINST the previous bar's direction (note the minus sign).
df['signal'] = -1 * df['close_log_return_lag_1_dir']
df

In [ ]:
df['trade_log_return'] = df['signal'] * df['close_log_return']
df

In [ ]:
df['trade_log_return'].cumsum().plot(title='Equity curve (gross, log returns)')

## Step 5 — How good is it?

**Win rate** — fraction of bars that made money.

In [ ]:
df['is_won'] = df['trade_log_return'] > 0
df['is_won'].mean()

**Total gross compound return** over the whole sample. Because our returns
are in log space, the compound return is `exp(sum) - 1`.

In [ ]:
r = np.exp(df['trade_log_return'].sum()) - 1
r

**Annualized return (CAGR).** The sample spans many days, so to compare it to
other strategies we annualize: grow the total return `r` at a constant daily rate
and scale to 365 days.

*(The original notebook used `12 * r` here, which only makes sense for a
one-month sample — the proper annualization below works for any length.)*

In [ ]:
n_days = len(df)
annualized_return = (1 + r) ** (365 / n_days) - 1
annualized_return

### Annualized Sharpe

Sharpe = mean / std of the per-bar return, scaled to a year. This dataset uses
**daily (`1d`)** bars, so there are `365` periods per year (crypto trades every
day) and we scale by `sqrt(365)`.

In [ ]:
df['trade_log_return'].mean() / df['trade_log_return'].std() * np.sqrt(365)

## Step 6 — Add round-trip fees

Every bar is a full **round trip** (entry + exit), and each side pays a fee in
**basis points** (1 bp = 0.01%). Exchanges charge a lower **maker** rate for
passive limit orders and a higher **taker** rate for aggressive market orders.
Here we assume we can post limit orders and pay the cheaper **maker** fee on both
sides. We track the dollar account value so the fee applies to the real notional
traded.

In [ ]:
# Equity curve in log space, then converted to a dollar account value.
df['cum_trade_log_return'] = df['trade_log_return'].cumsum()

capital = 1000
df['post_trade_notional_value'] = capital + df['cum_trade_log_return'] * capital
# Value entering each bar = previous bar's post value (seed the first with capital).
df['pre_trade_notional_value'] = df['post_trade_notional_value'].shift().fillna(capital)
df

In [ ]:
TAKER_FEE_BPS = 2.0   # aggressive (market order) — reference
MAKER_FEE_BPS = 1.5   # passive   (limit order) — what we assume we pay

def fee_bps(bp):
    """Convert a fee in basis points to a fraction (1 bp = 0.01% = 0.0001)."""
    return bp / 10_000

TAKER_FEE = fee_bps(TAKER_FEE_BPS)
MAKER_FEE = fee_bps(MAKER_FEE_BPS)

# Maker fee on both the entry (pre-trade notional) and the exit (post-trade).
df['entry_fee'] = df['pre_trade_notional_value'] * MAKER_FEE
df['exit_fee'] = df['post_trade_notional_value'] * MAKER_FEE
df['roundtrip_fees'] = df['entry_fee'] + df['exit_fee']
df['cum_roundtrip_fees'] = df['roundtrip_fees'].cumsum()
df

In [ ]:
# Net equity = gross account value minus the fees paid so far.
df['net_equity'] = df['post_trade_notional_value'] - df['cum_roundtrip_fees']
df

### Gross vs net

Comparing the two curves shows how much of the edge the fees consume — critical
for a strategy that turns over on every bar.

In [ ]:
ax = df['post_trade_notional_value'].plot(label='gross', legend=True)
df['net_equity'].plot(ax=ax, label='net (after fees)', legend=True, title='Equity: gross vs net ($)')

## Exercises

1. **Position sizing** — start from a fixed stake (say $12) and compound it bar to
   bar instead of using a constant $1000.
2. **Maker vs taker** — re-run the fees with the taker rate. Does the edge survive
   paying the aggressive fee on both sides?
3. **Signal depth** — try reverting against the last *two* bars instead of one.
4. **Compare** — run the [momentum](01-momentum.ipynb) notebook on this same asset.
   Does BCH reward fading moves or following them?